In [68]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# 데이터 로드
train_df = pd.read_csv(
    '../data/creditcard.csv'
)



# 답 피처 구분
y_labels=train_df.iloc[:, -1].copy()
X_features= train_df.drop(columns=['Class']).copy()




In [69]:
# 트레인 

X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_labels,
    test_size=0.2,
    stratify=y_labels,
    random_state=42
)
print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nTrain Fraud ratio:")
print(y_train.value_counts(normalize=True))

print("\nTest Fraud ratio:")
print(y_test.value_counts(normalize=True))

Train: (227845, 30)
Test : (56962, 30)

Train Fraud ratio:
Class
0    0.998271
1    0.001729
Name: proportion, dtype: float64

Test Fraud ratio:
Class
0    0.99828
1    0.00172
Name: proportion, dtype: float64


In [70]:
train_check = X_train.copy()
test_check = X_test.copy()

train_rows = set(
    map(tuple, train_check.to_numpy())
)

test_rows = set(
    map(tuple, test_check.to_numpy())
)

overlap = train_rows & test_rows

print(
    "Train/Test 완전 동일 Feature 행:",
    len(overlap)
)

Train/Test 완전 동일 Feature 행: 293


In [71]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# 데이터 로드
train_df = pd.read_csv(
    '../data/creditcard.csv'
)

print("중복 제거 전:", len(train_df))

train_df = train_df.drop_duplicates().reset_index(drop=True)

print("중복 제거 후:", len(train_df))

# 답 피처 구분
y_labels=train_df.iloc[:, -1].copy()
X_features= train_df.drop(columns=['Class']).copy()




중복 제거 전: 284807
중복 제거 후: 283726


In [73]:
#트레인 

X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_labels,
    test_size=0.2,
    stratify=y_labels,
    random_state=42
)
print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nTrain Fraud ratio:")
print(y_train.value_counts(normalize=True))

print("\nTest Fraud ratio:")
print(y_test.value_counts(normalize=True))

Train: (226980, 30)
Test : (56746, 30)

Train Fraud ratio:
Class
0    0.998335
1    0.001665
Name: proportion, dtype: float64

Test Fraud ratio:
Class
0    0.998326
1    0.001674
Name: proportion, dtype: float64


In [74]:
train_check = X_train.copy()
test_check = X_test.copy()

train_rows = set(
    map(tuple, train_check.to_numpy())
)

test_rows = set(
    map(tuple, test_check.to_numpy())
)

overlap = train_rows & test_rows

print(
    "Train/Test 완전 동일 Feature 행:",
    len(overlap)
)

Train/Test 완전 동일 Feature 행: 0


In [75]:
# amount scaling
from sklearn.preprocessing import StandardScaler, RobustScaler

X_train = X_train.copy()
X_test = X_test.copy()

rob_scaler = RobustScaler()

# Train 데이터에서만 중앙값/IQR 학습
X_train['Amount_Scaled'] = rob_scaler.fit_transform(
    X_train[['Amount']]
)

# Test는 Train에서 배운 기준으로 transform만
X_test['Amount_Scaled'] = rob_scaler.transform(
    X_test[['Amount']]
)

# 기존 Time, Amount 제거
X_train.drop(columns=['Time', 'Amount'], inplace=True)
X_test.drop(columns=['Time', 'Amount'], inplace=True)

In [76]:

#이상치 제거

def get_outlier(df, column, weight=1.5):

    fraud = df[df['Class'] == 1][column]

    q25 = np.percentile(fraud.values, 25)
    q75 = np.percentile(fraud.values, 75)

    iqr = q75 - q25

    lowest = q25 - weight * iqr
    highest = q75 + weight * iqr

    outlier_index = fraud[
        (fraud < lowest) |
        (fraud > highest)
    ].index

    return outlier_index

In [77]:
# 데이터 합침.
train_df = X_train.copy()
train_df['Class'] = y_train

In [78]:
outlier_index = get_outlier(
    train_df,
    column='V14',
    weight=1.5
)

print("제거되는 Train 이상치:", len(outlier_index))

train_df.drop(
    index=outlier_index,
    inplace=True
)

제거되는 Train 이상치: 3


In [79]:
#사기 / 일반 분리
fraud_train = train_df[
    train_df['Class'] == 1
].copy()

normal_train = train_df[
    train_df['Class'] == 0
].copy()

fraud_count = len(fraud_train)

print("Fraud Train:", fraud_count)
print("Normal Train:", len(normal_train))

Fraud Train: 375
Normal Train: 226602


In [80]:
normal_chunk_size = fraud_count * 10

normal_train = normal_train.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

sample_datasets = []

for start in range(
    0,
    len(normal_train),
    normal_chunk_size
):

    normal_chunk = normal_train.iloc[
        start:start + normal_chunk_size
    ]

    if len(normal_chunk) < normal_chunk_size:
        break

    sampled_df = pd.concat([
        fraud_train,
        normal_chunk
    ])

    sampled_df = sampled_df.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    sample_datasets.append(sampled_df)

print(
    "만들어진 Sample 수:",
    len(sample_datasets)
)

만들어진 Sample 수: 60


In [81]:
xgb_models = []

for i, sampled_df in enumerate(sample_datasets):

    X_sample = sampled_df.drop(
        columns='Class'
    )

    y_sample = sampled_df['Class']

    model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='auc',
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_sample,
        y_sample
    )

    xgb_models.append(model)

print(
    "학습된 XGB 모델:",
    len(xgb_models)
)

학습된 XGB 모델: 60


In [82]:
#테스트는 예측으로 수행
xgb_pred_proba_list = []

for model in xgb_models:

    pred_proba = model.predict_proba(
        X_test
    )[:, 1]

    xgb_pred_proba_list.append(
        pred_proba
    )

In [83]:
#최종 프로우드 확률
xgb_mean_pred_proba = np.mean(
    xgb_pred_proba_list,
    axis=0
)

In [84]:
from sklearn.metrics import average_precision_score


roc_auc = roc_auc_score(
    y_test,
    xgb_mean_pred_proba
)

pr_auc = average_precision_score(
    y_test,
    xgb_mean_pred_proba
)

print(
    "ROC-AUC:",
    roc_auc
)

print(
    "PR-AUC:",
    pr_auc
)

ROC-AUC: 0.9773376973881633
PR-AUC: 0.8149935903655809


In [85]:
from sklearn.metrics import confusion_matrix, classification_report

final_pred = (
    xgb_mean_pred_proba >= 0.5
).astype(int)

print(
    confusion_matrix(
        y_test,
        final_pred
    )
)

print(
    classification_report(
        y_test,
        final_pred
    )
)

[[56576    75]
 [   16    79]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.51      0.83      0.63        95

    accuracy                           1.00     56746
   macro avg       0.76      0.92      0.82     56746
weighted avg       1.00      1.00      1.00     56746



In [114]:
# hyperopt 하이퍼 파라미터 튜닝

from hyperopt import hp


# max_depth는 4에서 15까지 1간격으로, min_child_weight는 1에서 6까지 1간격으로
# colsample_bytree는 0.5에서 0.95사이, learning_rate는 0.01에서 0.2사이 정규 분포된 값으로 검색.


xgb_search_space = {'max_depth': hp.quniform('max_depth', 4, 15, 1),
                    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
                    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 0.95),
                    'subsample': hp.uniform('subsample', 0.6, 1.0),
                    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2)
}
# model = XGBClassifier(
#         n_estimators=300,
#         max_depth=5,
#         learning_rate=0.05,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         eval_metric='auc',
#         random_state=42,
#         n_jobs=-1
#     )


In [115]:
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score


# 목적 함수 설정.
# 추후 fmin()에서 입력된 search_space값으로 XGBClassifier 교차 검증 학습 후 -1* roc_auc 평균 값을 반환.  
def objective_func(search_space):
    xgb_clf = XGBClassifier(n_estimators=100, max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            colsample_bytree=search_space['colsample_bytree'],
                            subsample=search_space['subsample'],
                            early_stopping_rounds=30, eval_metric='auc'
                           )

    # 3개 k-fold 방식으로 평가된 roc_auc 지표를 담는 list
    roc_auc_list= []
   
    # 3개 k-fold방식 적용
    kf = KFold(n_splits=5)
    # X_train을 다시 학습과 검증용 데이터로 분리
    for tr_index, val_index in kf.split(X_train):
        # kf.split(X_train)으로 추출된 학습과 검증 index값으로 학습과 검증 데이터 세트 분리
        X_tr, y_tr = X_train.iloc[tr_index], y_train.iloc[tr_index]
        X_val, y_val = X_train.iloc[val_index], y_train.iloc[val_index]
        # early stopping은 30회로 설정하고 추출된 학습과 검증 데이터로 XGBClassifier 학습 수행.
        xgb_clf.fit(X_tr, y_tr, 
                   eval_set=[(X_tr, y_tr), (X_val, y_val)])
   
        # 1로 예측한 확률값 추출후 roc auc 계산하고 평균 roc auc 계산을 위해 list에 결과값 담음.
        score = roc_auc_score(y_val, xgb_clf.predict_proba(X_val)[:, 1])
        roc_auc_list.append(score)
       
    # 3개 k-fold로 계산된 roc_auc값의 평균값을 반환하되,
    # HyperOpt는 목적함수의 최소값을 위한 입력값을 찾으므로 -1을 곱한 뒤 반환.
    return -1 * np.mean(roc_auc_list)


In [116]:
from hyperopt import fmin, tpe, Trials


trials = Trials()


# fmin()함수를 호출. max_evals지정된 횟수만큼 반복 후 목적함수의 최소값을 가지는 최적 입력값 추출.
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=100, # 최대 반복 횟수를 지정합니다.
            trials=trials, rstate=np.random.default_rng(seed=30))


print('best:', best)

[0]	validation_0-auc:0.96799	validation_1-auc:0.92776  
[1]	validation_0-auc:0.90167	validation_1-auc:0.82956  
[2]	validation_0-auc:0.91644	validation_1-auc:0.85342  
[3]	validation_0-auc:0.91969	validation_1-auc:0.85842  
[4]	validation_0-auc:0.93018	validation_1-auc:0.86814  
[5]	validation_0-auc:0.93823	validation_1-auc:0.87149  
[6]	validation_0-auc:0.95141	validation_1-auc:0.88758  
[7]	validation_0-auc:0.95782	validation_1-auc:0.89601  
[8]	validation_0-auc:0.96087	validation_1-auc:0.89839  
[9]	validation_0-auc:0.96194	validation_1-auc:0.89594  
[10]	validation_0-auc:0.96597	validation_1-auc:0.89729 
[11]	validation_0-auc:0.96831	validation_1-auc:0.89697 
[12]	validation_0-auc:0.97083	validation_1-auc:0.89879 
[13]	validation_0-auc:0.97313	validation_1-auc:0.90061 
[14]	validation_0-auc:0.97518	validation_1-auc:0.90135 
[15]	validation_0-auc:0.97707	validation_1-auc:0.90686 
[16]	validation_0-auc:0.97859	validation_1-auc:0.90772 
[17]	validation_0-auc:0.97967	validation_1-auc:0

In [117]:
for i, sampled_df in enumerate(sample_datasets):

    X_sample = sampled_df.drop(
        columns='Class'
    )

    y_sample = sampled_df['Class']

    # model = XGBClassifier(
    #     n_estimators=300,
    #     max_depth=5,
    #     learning_rate=0.05,
    #     subsample=0.8,
    #     colsample_bytree=0.8,
    #     eval_metric='auc',
    #     random_state=42,
    #     n_jobs=-1
    # )
    model= XGBClassifier(n_estimators=1000, learning_rate=round(best['learning_rate'], 5),
                        max_depth=int(best['max_depth']), min_child_weight=int(best['min_child_weight']),
                        eval_metric="auc",subsample=best['subsample'],
                        colsample_bytree=round(best['colsample_bytree'], 5),random_state=42,
                       n_jobs=-1  
                       )

    model.fit(
        X_sample,
        y_sample
    )

    xgb_models.append(model)

In [118]:
#테스트는 예측으로 수행
xgb_pred_proba_list = []

for model in xgb_models:

    pred_proba = model.predict_proba(
        X_test
    )[:, 1]

    xgb_pred_proba_list.append(
        pred_proba
    )

#최종 프로우드 확률
xgb_mean_pred_proba = np.mean(
    xgb_pred_proba_list,
    axis=0
)

In [119]:
roc_auc = roc_auc_score(
    y_test,
    xgb_mean_pred_proba
)

pr_auc = average_precision_score(
    y_test,
    xgb_mean_pred_proba
)

print(
    "ROC-AUC:",
    roc_auc
)

print(
    "PR-AUC:",
    pr_auc
)

ROC-AUC: 0.9706256869159182
PR-AUC: 0.8163500498533911


In [110]:
final_pred = (
    xgb_mean_pred_proba >= 0.5
).astype(int)

print(
    confusion_matrix(
        y_test,
        final_pred
    )
)

print(
    classification_report(
        y_test,
        final_pred
    )
)

[[56538   113]
 [   16    79]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.41      0.83      0.55        95

    accuracy                           1.00     56746
   macro avg       0.71      0.91      0.77     56746
weighted avg       1.00      1.00      1.00     56746

